# **MACRO ANALYSIS AND OPPORTUNITIES FOR LIFESCIENCE INDUSTRY EUROPE**

# 1. DATA EXTRACTION

The following work was made to analyze the opportunities in the European Market for companies in the Medtech.

**Source of Information**:

1) Company Reports: We get the financial reports of the company for the last three years.
2) CORDIS - EU research projects under HORIZON EUROPE (2021 - 2027)
3) Clinical Trias Information from *"https://clinicaltrials.gov/api/v2/studies"*

## 1. CORDIS - EU RESEARCH PROJECTS UNDER HORIZON EUROPE

### Libraries

In [ ]:
import sys
!{sys.executable} -m pip install openpyxl
import zipfile
import requests
import urllib.request
import pandas as pd
from pathlib import Path
import os
import re
os.getcwd()

### 2014 - 2020 Dataset

In [ ]:
#url_2014_2020="https://cordis.europa.eu/data/cordis-h2020projects-xlsx.zip"
reports_dir= Path("data/Reports")
#reports_dir.mkdir(parents=True, exist_ok=True)
#zip_path = reports_dir / "cordis-h2020projects-xlsx.zip"
#urllib.request.urlretrieve(url_2014_2020, zip_path)
#with zipfile.ZipFile(zip_path, 'r') as z:
#    z.extract("project.xlsx",reports_dir)
#    z.extract("organization.xlsx",reports_dir)
    
#(reports_dir / "project.xlsx").rename(reports_dir/"project_2014_2020.xlsx")
#(reports_dir / "organization.xlsx").rename(reports_dir/"organization_2014_2020.xlsx")

In [ ]:
df_eu_org_14_20 = pd.read_excel(reports_dir / "organization_2014_2020.xlsx", engine="openpyxl")
df_eu_proj_14_20 = pd.read_excel(reports_dir / "project_2014_2020.xlsx", engine="openpyxl")


### 2021 - 2027 Dataset

In [ ]:
#url_2021_2027="https://cordis.europa.eu/data/cordis-HORIZONprojects-xlsx.zip"
reports_dir= Path("data/Reports")
#reports_dir.mkdir(parents=True, exist_ok=True)
#zip_path = reports_dir / "cordis-HORIZONprojects-xlsx.zip"
#with zipfile.ZipFile(zip_path, 'r') as z:
#urllib.request.urlretrieve(url_2021_2027, zip_path)
#    z.extract("project.xlsx",reports_dir)
#    z.extract("organization.xlsx",reports_dir)
    
#(reports_dir / "project.xlsx").rename(reports_dir/"project_2021_2027.xlsx")
#(reports_dir / "organization.xlsx").rename(reports_dir/"organization_2021_2027.xlsx")

In [ ]:
df_eu_org_21_27=pd.read_excel(reports_dir/"organization_2021_2027.xlsx", engine='openpyxl')
df_eu_proj_21_27=pd.read_excel(reports_dir/"project_2021_2027.xlsx", engine='openpyxl')

### Understanding the Data Structure

In [ ]:
df_eu_org_14_20.head()

In [ ]:
df_eu_org_21_27.head()

In [ ]:
df_eu_proj_14_20.head()

In [ ]:
df_eu_proj_21_27.head()

In [ ]:
df_eu_org_14_20.columns

In [ ]:
df_eu_org_21_27.columns

In [ ]:
common_eu_org_cols=set(df_eu_org_14_20.columns).intersection(set(df_eu_org_21_27.columns))
common_eu_org_cols

In [ ]:
df_eu_proj_14_20.columns

In [ ]:
df_eu_proj_21_27.columns

In [ ]:
common_eu_proj_cols=set(df_eu_proj_14_20.columns).intersection(set(df_eu_proj_21_27.columns))
common_eu_proj_cols

In [ ]:
df_eu_proj_14_20.rename(columns={'id':'projectID'}, inplace=True)
df_eu_proj_14_20.columns

In [ ]:
df_eu_proj_21_27.rename(columns={'id':'projectID'}, inplace=True)
df_eu_proj_21_27.columns

In [ ]:
common_cols=set(df_eu_proj_14_20.columns).intersection(set(df_eu_org_14_20.columns))
common_cols

In [ ]:
common_cols=set(df_eu_proj_21_27.columns).intersection(set(df_eu_org_21_27.columns))
common_cols

In [ ]:
df_eu_org_14_20.info()

In [ ]:
df_eu_proj_14_20.info()

In [ ]:
df_eu_org_21_27.info()

In [ ]:
df_eu_proj_21_27.info()

In [ ]:
numeric_cols=["totalCost","totalContribution","ecMaxContribution","ecContribution","ecContributionEUR"]
date_cols=["startDate","endDate"]

for df in [df_eu_org_14_20, df_eu_proj_14_20, df_eu_org_21_27, df_eu_proj_21_27]:
    for col in numeric_cols:
        if col in df.columns:
            df[col]= (
                df[col]
                .astype(str)
                .str.replace(",",".",regex=False)
                .replace(["nan","None", ""],pd.NA)
            )
            df[col]=pd.to_numeric(df[col],errors="coerce")


for df in [df_eu_org_14_20, df_eu_proj_14_20, df_eu_org_21_27, df_eu_proj_21_27]:
    for col in date_cols:
        if col in df.columns:
            df[col]=pd.to_datetime(df[col],errors="coerce")

In [ ]:
df_eu_org_14_20["totalCost"].info()

In [ ]:
## Check if both columns have the same values 2014-2020 dataset
org_tot_cost_14_20= df_eu_org_14_20['totalCost'].sum()
proj_tot_cost_14_20= df_eu_proj_14_20['totalCost'].sum()

print(org_tot_cost_14_20)
print(proj_tot_cost_14_20)


In [ ]:
## Check if both columns have the same values 2021-2027
org_tot_cost_21_27= df_eu_org_21_27['totalCost'].sum()
proj_tot_cost_21_27= df_eu_proj_21_27['totalCost'].sum()

print(org_tot_cost_21_27)
print(proj_tot_cost_21_27)

After checking the information, the `Project` dataset is the one that contains the unique information related to each project, while the `Organizations` dataset brings the information related to all the laboratories and entities that are part of the projects.

Because the idea of this project is to understand the opportunity of potential customers, we will work with both datasets with major focus in Organizations.

We also realized that the cost information is not completely exact in both files. For example, in the `Project` file the total cost could be EUR 5,000, while in the `Organizations` file the cost could appear as EUR 5,010. Also the Total Cost column in the project table is related to the total cost of the project, while the organization total cost is the cost that was assign to the organization.

For this analysis, we will focus on the `totalCost` from the `Organizations` table.


Following steps:

1. Take the `Project` table and filter all the projects related to the MedTech field of the company.
2. Based on this filtered `Project` table, filter the projects in the `Organizations` table to check only the relevant entities for the company.

### Target Projects - Focus on Health & Medtech

In [ ]:
keywords= [
    "laboratory automation", "lab automation",
    "liquid handling", "sample preparation", "workflow automation",    

    # Genomics / molecular biology
    "genomics", "genomic", "proteomics", "proteomic",
    "multi-omics", "multiomics", "transcriptomics", "metabolomics",
    "single cell", "sequencing", "next generation sequencing", "ngs",
    "pcr", "qpcr", "digital pcr", "rna", "dna",

    # Biomarkers / diagnostics
    "biomarker", "biomarkers",
    "diagnostics",  "molecular diagnostics",
    "clinical diagnostics", "ivd", "in vitro diagnostics",
    "liquid biopsy", "companion diagnostics",
    "genetic testing", "molecular profiling",
    "pathology", "molecular pathology",

    # Oncology / high-value clinical areas
    "oncology", "cancer", "tumor", "tumour",
    "leukemia", "leukaemia", "lymphoma", "carcinoma",
    "solid tumor", "solid tumour",

    # Cell / tissue / advanced therapies
    "cell therapy", "cart", "car-t", "t cell", "t-cell",
    "stem cell", "cell biology", "cellomics",
    "organoids", "3d cell", "cell and tissue",

    # Pharma / biotech
    "drug discovery", "drug development", "biopharma",
    "biotechnology", "personalized medicine", "precision medicine",

    # Digital / AI
    "bioinformatics", "data-driven", "digital health",

    # MedTech / OEM
    "medtech", "medical device", 
    "microfluidics"
]

In [ ]:
def filter_projects_by_keywords(df, text_col="objective"):
    pattern = "|".join(re.escape(k.lower()) for k in keywords)
    return df[
        df[text_col]
        .fillna("")
        .str.lower()
        .str.contains(pattern, regex=True)
    ].copy()
    

In [ ]:
df_eu_proj_14_20_final= filter_projects_by_keywords(df_eu_proj_14_20, text_col="objective")
df_eu_proj_21_27_final= filter_projects_by_keywords(df_eu_proj_21_27, text_col="objective")

In [ ]:
df_eu_proj_14_20.info()

In [ ]:
df_eu_proj_14_20_final.info()

 Based on this filter we will filter the unique project for the `Organization` table and to analyze the opportunity there.

In [ ]:
def build_opportunity_base (df_org, df_proj_filtered, proj_columns):
    """
    Build the opportunity base:
    1) Keep inly organizations linked to filtered project based on keywords.
    2) Add selected project-level columns
    """
    
    assert df_proj_filtered["projectID"].is_unique,"ProjectID is not unique in project table"
    
    df_org_filtered = df_org[df_org["projectID"].isin(df_proj_filtered["projectID"])].copy()
    
    df_final=df_org_filtered.merge(
        df_proj_filtered[proj_columns],
        how="left",
        on="projectID",
        validate="many_to_one"
    )
    
    return df_final
    

In [ ]:
proj_columns = ['projectID','status', 'title', 'startDate','endDate','totalCost','topics','fundingScheme','grantDoi','keywords','objective']

In [ ]:
df_eu_org_14_20_final=build_opportunity_base(
    df_org=df_eu_org_14_20,
    df_proj_filtered=df_eu_proj_14_20_final,
    proj_columns=proj_columns
)

In [ ]:
df_eu_org_14_20_final.rename(columns={'totalCost_x':'totalCostOrg','totalCost_y':'totalCostProj'},inplace=True)
df_eu_org_14_20_final.columns

In [ ]:
df_eu_org_21_27_final=build_opportunity_base(
    df_org=df_eu_org_21_27,
    df_proj_filtered= df_eu_proj_21_27_final,
    proj_columns=proj_columns
)

In [ ]:
df_eu_org_21_27_final.rename(columns={'totalCost_x':'totalCostOrg','totalCost_y':'totalCostProj'},inplace=True)
df_eu_org_21_27_final.columns

Now we will consolidate both tables, and adding the `period` variable for each:

In [ ]:
df_eu_org_14_20_final["period"] = "2014-2020"
df_eu_org_21_27_final["period"] = "2021-2027"

In [ ]:
df_eu_cordis_db = pd.concat([df_eu_org_14_20_final,df_eu_org_21_27_final],ignore_index=True)

In [ ]:
df_eu_cordis_db.columns

In [ ]:
df_eu_final_cols =['projectID','organisationID','name','shortName','activityType','city','country','role','ecContribution',
       'netEcContribution', 'totalCostOrg', 'endOfParticipation', 'active',
       'status', 'title', 'startDate', 'endDate', 'totalCostProj', 'topics',
       'fundingScheme', 'grantDoi', 'keywords', 'period','objective']

In [ ]:
df_eu_cordis_db = df_eu_cordis_db[df_eu_final_cols].copy()

In [ ]:
df_eu_cordis_db.shape

In [ ]:
df_eu_cordis_db["period"].value_counts()

And we have the first data set :)

In [ ]:
df_eu_cordis_db.to_csv("../../_local_data_backup/CordisDatabase.csv")

## 2. CLINICAL TRIAL DATASETS

Information download from: https://euclinicaltrials.eu/search-for-clinical-trials/?lang=en

In [169]:
os.getcwd()

'/Users/fernandacladeramelgar/Documents/2_Projects/TECAN_Application/SalesAnalytics/Scripts'

In [170]:
df_trials = pd.read_csv("data/data_raw/CTIS_trials_20260626.csv")
df_trials.head()


,Trial number,Protocol code,Title of the trial,Overall trial status,Location(s) and recruitment status,Age group,Age range secondary identifier,Gender,Number of participants enrolled,Trial region,...,Primary endpoint,Secondary endpoints,Decision date,Start date,End date,Global end of the trial,Trial results,Sponsor/Co-Sponsors,Sponsor type,Last updated
0,2023-509723-41-00,REALL_ CART,A Phase I Clinical Trial of CART cell therapy ...,Not authorised,Spain:Not authorised,"18-64 years, 0-17 years",NaN,"Female, Male",10,EEA only,...,"In this umbrella trial, we will determine in p...",Expression of CD19/CD22 and NKG2DL on primary ...,07/05/2024,NaN,NaN,NaN,No,Fundacion Para La Investigacion Biomedica Del ...,Patient organisation/association,07/05/2024
1,2023-510220-72-00,BRONCHIPHARMA I,PHARMACOLOGICAL STRATEGIES OF INHALED ANTIBIOT...,"Authorised, recruitment pending","Spain:Authorised, recruitment pending","65+ years, 18-64 years","65-84 years, 85+ years","Female, Male",75,EEA only,...,Number of colony forming units (CFU),NaN,03/06/2024,NaN,NaN,NaN,No,Consorci Mar Parc De Salut De Barcelona,Hospital/Clinic/Other health care facility,03/06/2024
2,2022-501417-31-00,MK-7684A-010,"A Phase 3, Randomized, Double-blind, Active-Co...",Not authorised,"Poland:Not authorised, Italy:Not authorised, G...","18-64 years, 65+ years, 0-17 years","12-17 years, 65-84 years, 85+ years, N/A","Female, Male",510,In both EEA and non-EEA,...,Recurrence-Free Survival (RFS),"Distant Metastasis-Free Survival (DMFS), Overa...",02/03/2023,NaN,NaN,NaN,No,Merck Sharp & Dohme LLC,Pharmaceutical company,06/03/2023
3,2024-512642-42-00,NaN,Randomized study to protect from radiation iat...,"Ongoing, recruiting","Italy:Ongoing, recruiting","65+ years, 0-17 years, 18-64 years",NaN,"Female, Male",65,EEA only,...,Hypothyroidism-free survival at 3 years after ...,NaN,24/05/2024,13/12/2021,NaN,NaN,No,Fondazione IRCCS Istituto Nazionale Dei Tumori,Hospital/Clinic/Other health care facility,24/05/2024
4,2023-504664-42-02,IP-VITD,REQUIRED DOSE OF CHOLECALCIFEROL IN THE MANAGE...,"Ongoing, recruiting","Spain:Ongoing, recruiting","65+ years, 18-64 years",NaN,Female,80,EEA only,...,"To evaluate the efficacy of cholecalciferol, s...",Vitamin D levels will be evaluated in the four...,27/06/2023,03/07/2023,NaN,NaN,No,Clinica Palacios Madrid S.L.,Hospital/Clinic/Other health care facility,27/06/2023


In [171]:
df_trials.columns

Index(['Trial number', 'Protocol code', 'Title of the trial',
       'Overall trial status', 'Location(s) and recruitment status',
       'Age group', 'Age range secondary identifier', 'Gender',
       'Number of participants enrolled', 'Trial region', 'Medical conditions',
       'Therapeutic area', 'Trial phase', 'Product', 'Primary endpoint',
       'Secondary endpoints', 'Decision date', 'Start date', 'End date',
       'Global end of the trial', 'Trial results', 'Sponsor/Co-Sponsors',
       'Sponsor type', 'Last updated'],
      dtype='str')

In [172]:
df_trials.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                              Non-Null Count  Dtype
---  ------                              --------------  -----
 0   Trial number                        10000 non-null  str  
 1   Protocol code                       8463 non-null   str  
 2   Title of the trial                  10000 non-null  str  
 3   Overall trial status                10000 non-null  str  
 4   Location(s) and recruitment status  10000 non-null  str  
 5   Age group                           10000 non-null  str  
 6   Age range secondary identifier      3623 non-null   str  
 7   Gender                              10000 non-null  str  
 8   Number of participants enrolled     10000 non-null  int64
 9   Trial region                        9999 non-null   str  
 10  Medical conditions                  9997 non-null   str  
 11  Therapeutic area                    10000 non-null  str  
 12  Trial phase     

In [173]:
df_trials.columns = (
    df_trials.columns
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)
df_trials.columns

Index(['Trial_number', 'Protocol_code', 'Title_of_the_trial',
       'Overall_trial_status', 'Location(s)_and_recruitment_status',
       'Age_group', 'Age_range_secondary_identifier', 'Gender',
       'Number_of_participants_enrolled', 'Trial_region', 'Medical_conditions',
       'Therapeutic_area', 'Trial_phase', 'Product', 'Primary_endpoint',
       'Secondary_endpoints', 'Decision_date', 'Start_date', 'End_date',
       'Global_end_of_the_trial', 'Trial_results', 'Sponsor_Co-Sponsors',
       'Sponsor_type', 'Last_updated'],
      dtype='str')

In [174]:
date_cols_trials = [
    "Decision_date",
    "Start_date",
    "End_date",
    "Global_end_of_the_trial",
    "Last_updated"
]

for col in date_cols_trials:
    df_trials[col]=pd.to_datetime(df_trials[col],errors="coerce")
    

/var/folders/_z/vq1q1pzj639b28jbhlxzd0zw0000gn/T/ipykernel_21635/2708697362.py:10: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_trials[col]=pd.to_datetime(df_trials[col],errors="coerce")
/var/folders/_z/vq1q1pzj639b28jbhlxzd0zw0000gn/T/ipykernel_21635/2708697362.py:10: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_trials[col]=pd.to_datetime(df_trials[col],errors="coerce")
/var/folders/_z/vq1q1pzj639b28jbhlxzd0zw0000gn/T/ipykernel_21635/2708697362.py:10: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_trials[col]=pd.to_datetime(df_trials[col],errors="coerce")


In [175]:
df_trials.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Trial_number                        10000 non-null  str           
 1   Protocol_code                       8463 non-null   str           
 2   Title_of_the_trial                  10000 non-null  str           
 3   Overall_trial_status                10000 non-null  str           
 4   Location(s)_and_recruitment_status  10000 non-null  str           
 5   Age_group                           10000 non-null  str           
 6   Age_range_secondary_identifier      3623 non-null   str           
 7   Gender                              10000 non-null  str           
 8   Number_of_participants_enrolled     10000 non-null  int64         
 9   Trial_region                        9999 non-null   str           
 10  Medical_conditions                

In [176]:
search_cols = [
    "Title_of_the_trial",
    "Overall_trial_status",
    "Location(s)_and_recruitment_status",
    "Trial_region",
    "Medical_conditions",
    "Therapeutic_area",
    "Trial_phase",
    "Product",
    "Primary_endpoint",
    "Secondary_endpoints",
    "Trial_results",
    "Sponsor_Co-Sponsors",
    "Sponsor_type",
]

keep_cols = search_cols + [
    "Decision_date",
    "Start_date",
    "End_date",
]

In [177]:
def filter_trials_by_keywords(df, keywords, search_cols, keep_cols):
    pattern = "|".join(re.escape(k.lower()) for k in keywords)
    combined_text = (
        df[search_cols]
        .fillna("")
        .astype("string")
        .agg(lambda row: " ".join(row.dropna().astype(str)), axis=1)
        .str.lower()
    )
    mask = combined_text.str.contains(pattern, regex=True, na=False)
    return df.loc[mask, keep_cols].copy()

In [178]:
df_trials_final = filter_trials_by_keywords(
    df=df_trials,
    keywords=keywords,
    search_cols=search_cols,
    keep_cols=keep_cols
)

In [179]:
df_trials.shape

(10000, 24)

In [180]:
df_trials_final.shape

(5230, 16)

In [181]:
df_trials_final.to_csv("../Scripts/data/data_final/TrialsDatabase.csv")